In [98]:
from stat import FILE_ATTRIBUTE_SPARSE_FILE
import pandas as pd
import numpy as np
import os 

# 1. Get current directory (works in Notebooks)
current_dir = os.getcwd()

# 2. Path: Go UP one level (../)
file_path = os.path.join(current_dir, "../data/final/master_accidents.csv")
# FILE_PATH = "../data/final/Master_Accident_Dataset.csv"

df = pd.read_csv(file_path)
print(df.shape)
print(df.columns)


(4473, 19)
Index(['accident_id', 'source', 'year', 'month', 'day_of_week', 'state',
       'vehicle_1', 'vehicle_2', 'vehicles_involved', 'party_age_group',
       'victim_age_group', 'cause_of_accident', 'weather_condition',
       'road_condition', 'lighting_condition', 'fatalities', 'injuries',
       'casualties', 'severity'],
      dtype='str')


In [99]:
print(df.dtypes)
df.head()
df.columns

accident_id             str
source                  str
year                  int64
month                   str
day_of_week             str
state                   str
vehicle_1               str
vehicle_2               str
vehicles_involved       str
party_age_group         str
victim_age_group        str
cause_of_accident       str
weather_condition       str
road_condition          str
lighting_condition      str
fatalities            int64
injuries              int64
casualties            int64
severity                str
dtype: object


Index(['accident_id', 'source', 'year', 'month', 'day_of_week', 'state',
       'vehicle_1', 'vehicle_2', 'vehicles_involved', 'party_age_group',
       'victim_age_group', 'cause_of_accident', 'weather_condition',
       'road_condition', 'lighting_condition', 'fatalities', 'injuries',
       'casualties', 'severity'],
      dtype='str')

# Understanding the distribution

In [100]:
df.describe()

,year,fatalities,injuries,casualties
count,4473.000000,4473.000000,4473.000000,4473.000000
mean,789.345853,1.049408,1.186899,3.124301
std,986.589570,2.549513,3.801826,4.940588
min,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,1.000000
50%,0.000000,0.000000,0.000000,1.000000
75%,2022.000000,1.000000,0.000000,3.000000
max,2026.000000,79.000000,69.000000,79.000000


In [105]:
df['state'].value_counts()
# 1. Define the Correction Dictionary
state_corrections = {
    # Fix Typos
    'Maharastra': 'Maharashtra',
    'Rajesthan': 'Rajasthan',
    'Arnataka': 'Karnataka',
    
    # Fix Abbreviations & Regions
    'J & K': 'Jammu and Kashmir',
    'Jammu': 'Jammu and Kashmir',  # Map city/region to UT
    'Jammu And Kashmir': 'Jammu and Kashmir', # Standardize capitalization if needed
    
    # Note: Delhi, Chandigarh, and Puducherry are Union Territories 
    # and are correctly treated as state-level entities in Indian datasets.
}

# 2. Apply the corrections
df['state'] = df['state'].replace(state_corrections)

# 3. Verify the final clean list
print(df['state'].value_counts().count())

33


## Imputing and Handling missing columns

In [106]:
df['year'].value_counts()

year
0       2727
2022     690
2023     590
2021      90
2025      83
2024      76
2020      74
2019      71
2018      62
2026      10
Name: count, dtype: int64

Observation: The year column in the Kaggle dataset appears to be synthetically generated for the sole purpose of machine learning training, as it lacks realistic chronological details.

In [107]:
# Studying Fatalities and Injuries columns 
quantile_99th_fatalities = df['fatalities'].quantile(0.99)
quantile_99th_injuries = df['injuries'].quantile(0.99)
print(quantile_99th_injuries)
print(quantile_99th_fatalities)
 
# these data could not be removed based on outlier as there are accidents where casualities are more than 10 hence data retained for sevearity prediction


20.0
8.279999999999745


In [108]:
# analyzing vehicle_1 
print(df['vehicle_1'].value_counts())

# List of vehicles to group
heavy_vehicles = ['Tractor', 'Lorry', 'Tanker', 'Container', 'Tipper', 'Trailer', 'Mini Truck', 'crane','volvo']

df['vehicle_1'] = df['vehicle_1'].replace(heavy_vehicles,'Truck')


vehicle_1
Truck             3342
Two Wheeler        403
Car                251
Bus                130
Pedestrian         130
Tractor             58
Suv                 45
Auto Rickshaw       37
Van                 29
Pick Up             12
Lorry               12
Bicycle              9
Ambulance            3
Tanker               3
Container            2
Jugaad Vehicle       1
Train                1
Tipper               1
Mini Truck           1
Volvo                1
Trailer              1
Crane                1
Name: count, dtype: int64


In [109]:
# analyzing vehicle_1 
print(df['vehicle_2'].value_counts())

# List of vehicles to group
heavy_vehicles = ['Tractor', 'Lorry', 'Tanker', 'Container', 'Tipper', 'Trailer', 'Mini Truck','Canter','crane','volvo']

df['vehicle_2'] = df['vehicle_2'].replace(heavy_vehicles,'Truck')
df['vehicle_2'] = df['vehicle_2'].replace(['Unknown','Nil','Others'],'Unidentified')
cars = ['Suv','Pick Up','Fortuner','Bolero','Jeep','Innova']
df['vehicle_2'] = df['vehicle_2'].replace(cars,'Car')
df['vehicle_2'] = df['vehicle_2'].replace(['Bike','Scooter','Two Wheeler','Motorcycle'],'Two_Wheeler')

vehicle_2
Unknown          3261
Truck             920
Tractor            82
Lorry              65
Nil                37
Car                33
Gorge              15
Bus                11
Bike               10
Auto                5
Suv                 3
Others              3
Van                 3
Canal               2
Unidentified        2
Pick Up             2
Ditch               2
Scooter             2
Wall                1
River               1
Ambulance           1
Divider             1
Two Wheeler         1
Auto Rickshaw       1
Bicycle             1
Fortuner            1
Bolero              1
Canter              1
Motorcycle          1
Innova              1
Trailer             1
Tanker              1
Jeep                1
Name: count, dtype: int64


In [111]:
print(df['vehicles_involved'].value_counts())

# Imputing the unknown with mode value as this is not going to be used only in visulaization
mode_value = df['vehicles_involved'].mode()[0]
df['vehicles_involved'] = df['vehicles_involved'].replace('Unknown', mode_value)
print(df['vehicles_involved'].value_counts())


vehicles_involved
2          1944
Unknown    1297
1           535
3           441
4           155
5            92
6             9
Name: count, dtype: int64
vehicles_involved
2    3241
1     535
3     441
4     155
5      92
6       9
Name: count, dtype: int64


In [110]:
print(df['party_age_group'].value_counts())
print(df['victim_age_group'].value_counts())
df['victim_age_group'] = df['victim_age_group'].replace(['19-30'],'18-30')
df['victim_age_group'] = df['victim_age_group'].replace(['31-45'],'31-50')
df['victim_age_group'] = df['victim_age_group'].replace(['0-18'],'Under 18')

party_age_group
Unknown     1633
31-50       1107
18-30       1065
Over 51      488
Under 18     180
Name: count, dtype: int64
victim_age_group
Unknown     2318
18-30        802
31-50        715
Under 18     350
Over 51      279
0-18           7
31-45          1
19-30          1
Name: count, dtype: int64


In [112]:
df['cause_of_accident'].value_counts()

# 1. Define the Semantic Mapping Dictionary
cause_mapping = {
    # Lane Violations
    'Changing Lane To The Right': 'Lane Violation',
    'Changing Lane To The Left': 'Lane Violation',
    'Driving To The Left': 'Lane Violation',
    'Overtaking': 'Lane Violation',

    # Distancing & Rear-End (Often related)
    'No Distancing': 'No Distancing/Rear End',
    'Hit From Back': 'No Distancing/Rear End',
    'Moving Backward': 'No Distancing/Rear End',

    # Reckless & Rules
    'Driving Carelessly': 'Reckless Driving',
    'No Priority To Vehicle': 'Reckless Driving',
    'No Priority To Pedestrian': 'Reckless Driving',
    'Hit And Run': 'Reckless Driving',
    'Getting Off The Vehicle Improperly': 'Reckless Driving',
    
    # Severe Collisions
    'Head On Collision': 'Severe Collision',
    'Hit From Side': 'Severe Collision',
    'Collision': 'Severe Collision',
    'Fixed Object Collision': 'Severe Collision',

    # Speeding
    'Overspeeding': 'Speeding',
    'Driving At High Speed': 'Speeding',
    'Overspeed': 'Speeding',

    # Loss of Control
    'Overturning': 'Loss of Control',
    'Run Off Road': 'Loss of Control',
    'Vehicle Overturn': 'Loss of Control',
    'Turnover': 'Loss of Control',
    'Overloading': 'Loss of Control',

    # Parking
    'With Parked Vehicle': 'Parking Issue',
    'Improper Parking': 'Parking Issue',

    # DUI
    'Driving Under The Influence Of Drugs': 'Drunk/Drug Driving',
    'Drunk Driving': 'Drunk/Drug Driving',

    # Generalize Unknowns
    'Others': 'Unknown/Other',
    'Other': 'Unknown/Other',
    'Unknown': 'Unknown/Other'
}

# 2. Apply the Mapping
df['cause_of_accident'] = df['cause_of_accident'].replace(cause_mapping)

# 3. Handle the "Noise" (News Headlines & Rare Events)
# Any cause that appears less than 5 times is likely a scraping error/headline
counts = df['cause_of_accident'].value_counts()
rare_labels = counts[counts < 5].index.tolist()

# Replace those rare labels with 'Unknown/Other'
df['cause_of_accident'] = df['cause_of_accident'].replace(rare_labels, 'Unknown/Other')

# 4. Verify the Final 10 Categories
print(df['cause_of_accident'].value_counts())

cause_of_accident
No Distancing/Rear End    997
Reckless Driving          905
Lane Violation            880
Unknown/Other             816
Severe Collision          405
Loss of Control           151
Parking Issue             137
Speeding                   96
Drunk/Drug Driving         86
Name: count, dtype: int64


In [113]:
df['weather_condition'].value_counts()

# 1. Calculate the Mode (Most common weather)
weather_mode = df['weather_condition'].mode()[0]
print(f"Mode Weather: {weather_mode}")

# 2. Impute 'Unknown' and 'Other' with the Mode
df['weather_condition'] = df['weather_condition'].replace(['Unknown', 'Other'], weather_mode)

# 3. Standardize Categories (Grouping similar weather)
weather_mapping = {
    # Rain Group
    'Rainy': 'Raining',
    'Raining And Windy': 'Raining',
    
    # Fog/Visibility Group
    'Hazy': 'Foggy',
    'Fog Or Mist': 'Foggy',
    
    # Wind/Storm Group
    'Stormy': 'Windy/Stormy',
    'Windy': 'Windy/Stormy',
    
    # Good Conditions 
    'Clear': 'Normal'
}

# Apply the mapping
df['weather_condition'] = df['weather_condition'].replace(weather_mapping)

# 4. Verify the final clean list
print(df['weather_condition'].value_counts())

Mode Weather: Normal
weather_condition
Normal          3697
Raining          435
Foggy            188
Windy/Stormy     110
Cloudy            31
Snow              12
Name: count, dtype: int64


In [114]:

# 1. Standardize Categories (Merging Duplicates first)
road_mapping = {
    'Wet Or Damp': 'Wet',
    'Wet': 'Wet',
    'Under Construction': 'Damaged/Construction',
    'Damaged': 'Damaged/Construction',
    'Snow': 'Wet' # Snow is rare in india
}
df['road_condition'] = df['road_condition'].replace(road_mapping)

# 2. Identify the Top 2 Categories for Scattering
top_1 = 'Dry'
top_2 = 'Wet'

# Calculate their counts in the KNOWN data
count_1 = df[df['road_condition'] == top_1].shape[0]
count_2 = df[df['road_condition'] == top_2].shape[0]
total_known = count_1 + count_2

# Calculate Probabilities (Weights)
prob_1 = count_1 / total_known
prob_2 = count_2 / total_known

print(f"Scattering Logic: {top_1} ({prob_1:.2%}) vs {top_2} ({prob_2:.2%})")

# 3. Apply Scattering to 'Unknown' values
# Find the rows that are 'Unknown'
mask = df['road_condition'] == 'Unknown'
missing_count = mask.sum()

# Generate random choices based on the calculated probabilities
imputed_values = np.random.choice([top_1, top_2], size=missing_count, p=[prob_1, prob_2])

# Fill them in
df.loc[mask, 'road_condition'] = imputed_values

# 4. Verify the Result
print(df['road_condition'].value_counts())



Scattering Logic: Dry (72.48%) vs Wet (27.52%)
road_condition
Dry                     3054
Wet                     1194
Damaged/Construction     225
Name: count, dtype: int64


In [115]:
df['lighting_condition'].value_counts()

import numpy as np

# 1. Standardize Categories (Group all "Darkness" variants together)
lighting_mapping = {
    # Merge all Dark variants into one 'Dark' bucket
    'Darkness - Lights Lit': 'Dark',
    'Darkness - No Lighting': 'Dark',
    'Darkness - Lights Unlit': 'Dark',
    
    # Merge Dawn/Dusk into 'Twilight' (Distinct from Day/Dark)
    'Dusk': 'Twilight',
    'Dawn': 'Twilight'
}
df['lighting_condition'] = df['lighting_condition'].replace(lighting_mapping)

# 2. Identify the Top 2 Categories for Scattering
# Daylight (~2043) vs Dark (~900) are the top 2
top_1 = 'Daylight'
top_2 = 'Dark'

# Calculate their counts in the KNOWN data
count_1 = df[df['lighting_condition'] == top_1].shape[0]
count_2 = df[df['lighting_condition'] == top_2].shape[0]
total_known = count_1 + count_2

# Calculate Probabilities
prob_1 = count_1 / total_known
prob_2 = count_2 / total_known

print(f"Scattering Logic: {top_1} ({prob_1:.2%}) vs {top_2} ({prob_2:.2%})")

# 3. Apply Scattering to 'Unknown' values
mask = df['lighting_condition'] == 'Unknown'
missing_count = mask.sum()

# Generate random choices
imputed_values = np.random.choice([top_1, top_2], size=missing_count, p=[prob_1, prob_2])

# Fill them in
df.loc[mask, 'lighting_condition'] = imputed_values

# 4. Verify the Result
print(df['lighting_condition'].value_counts())

Scattering Logic: Daylight (69.37%) vs Dark (30.63%)
lighting_condition
Daylight    2935
Dark        1307
Twilight     231
Name: count, dtype: int64


In [116]:
df['severity'].value_counts()

severity
Minor      2511
Fatal      1423
Serious     539
Name: count, dtype: int64

In [119]:
print(df['fatalities'].value_counts())
print(df['injuries'].value_counts())

fatalities
0     2860
1      597
2      322
3      241
4      181
5      140
6       52
7       22
8       13
10      10
9        8
12       5
13       4
11       3
15       3
19       2
16       1
79       1
50       1
51       1
17       1
26       1
29       1
20       1
25       1
23       1
Name: count, dtype: int64
injuries
0     3507
1      265
2      138
3       96
4       82
5       68
7       52
6       51
8       40
10      31
9       22
12      16
20      15
11      13
14      10
15       8
13       8
17       7
18       6
22       5
19       4
16       3
24       3
40       3
27       3
34       2
23       2
36       2
30       2
31       1
26       1
69       1
41       1
28       1
45       1
25       1
49       1
21       1
Name: count, dtype: int64


#### Dropping casualities and Year columns as it is not going to help model prediction.

In [138]:
df_final = df.copy()

In [140]:
df_final.to_csv("../data/final/accident_ml_data.csv")